In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)

In [2]:
# Load combined CPU / host metrics dataset (all hosts in one file)
host_metrics = pd.read_csv(
    r"C:\Users\Avilasha\Desktop\CPU_Predictive_Maintenance\CPU\data\inference_data\orion db data(host_metrics).csv"
)
# Load combined iLO server metrics dataset (all hosts in one file)
ilo_metrics = pd.read_csv(
    r"C:\Users\Avilasha\Desktop\CPU_Predictive_Maintenance\CPU\data\inference_data\orion db data(ilo_server_metrics).csv",
    usecols=[
        "ts",
        "ilo_server_id",
        "inlet_temp_c",
        "cpu_temp_c",
        "processor_count"
    ]
)

In [3]:
# inspect datasets
print("host_metrics")
display(host_metrics.head())

print("ilo_metrics")
display(ilo_metrics.head())

# Check column names
print(host_metrics.columns)
print(ilo_metrics.columns)

host_metrics


,id,ts,host_id,cpu_usage_pct,memory_usage_pct,power_kw,temperature_c,status
0,1,4/2/2026 12:22,1,4.0,74.0,0.220,20.0,Normal
1,2,4/2/2026 12:22,2,17.0,70.0,0.245,20.0,Normal
2,3,4/2/2026 12:22,3,28.0,27.0,0.253,18.0,Normal
3,4,4/2/2026 12:24,1,5.0,74.0,0.220,20.0,Normal
4,5,4/2/2026 12:24,2,16.0,70.0,0.263,20.0,Normal


ilo_metrics


,ts,ilo_server_id,inlet_temp_c,cpu_temp_c,processor_count
0,4/2/2026 12:22,1,20.0,38.0,2
1,4/2/2026 12:22,2,20.0,43.0,2
2,4/2/2026 12:22,3,18.0,28.0,2
3,4/2/2026 12:24,1,20.0,38.0,2
4,4/2/2026 12:24,2,20.0,44.0,2


Index(['id', 'ts', 'host_id', 'cpu_usage_pct', 'memory_usage_pct', 'power_kw',
       'temperature_c', 'status'],
      dtype='object')
Index(['ts', 'ilo_server_id', 'inlet_temp_c', 'cpu_temp_c', 'processor_count'], dtype='object')


In [4]:
# convert timestamp columns
host_metrics["ts"] = pd.to_datetime(
    host_metrics["ts"],
    format="mixed",
    utc=True
)
ilo_metrics["ts"] = pd.to_datetime(
    ilo_metrics["ts"],
    format="mixed",
    utc=True
)

In [5]:
# verify dataset size
print(host_metrics.shape)
print(ilo_metrics.shape)

print("\nUnique host_id values:", sorted(host_metrics["host_id"].unique()))
print("Unique ilo_server_id values:", sorted(ilo_metrics["ilo_server_id"].unique()))

(92793, 8)
(92793, 5)

Unique host_id values: [np.int64(1), np.int64(2), np.int64(3)]
Unique ilo_server_id values: [np.int64(1), np.int64(2), np.int64(3), np.int64(72622)]


In [7]:
# sanity check for corrupted ilo_server_id rows (won't match on merge, but good to flag)
bad_ilo_rows = ilo_metrics[~ilo_metrics["ilo_server_id"].isin([1, 2, 3])]
print(f"Rows with invalid ilo_server_id: {len(bad_ilo_rows)}")
if len(bad_ilo_rows) > 0:
    display(bad_ilo_rows)

# merge CPU metrics with iLO metrics on timestamp + host id
cpu_master_new = pd.merge(
    host_metrics,
    ilo_metrics,
    left_on=["ts", "host_id"],
    right_on=["ts", "ilo_server_id"],
    how="inner"
)
print(cpu_master_new.shape)

Rows with invalid ilo_server_id: 1


,ts,ilo_server_id,inlet_temp_c,cpu_temp_c,processor_count
72621,2026-06-09 06:26:00+00:00,72622,NaN,NaN,0


(92792, 12)


In [8]:
# merge CPU metrics with iLO metrics on timestamp + host id
cpu_master_new = pd.merge(
    host_metrics,
    ilo_metrics,
    left_on=["ts", "host_id"],
    right_on=["ts", "ilo_server_id"],
    how="inner"
)

print(cpu_master_new.shape)

(92792, 12)


In [9]:
# map host_id -> hostName
HOST_MAP = {
    1: "10.10.10.65",
    2: "10.10.10.150",
    3: "10.10.10.2"
}
cpu_master_new["hostName"] = (
    cpu_master_new["host_id"]
    .map(HOST_MAP)
)
cpu_master_new.shape
# inspect combined dataset
display(cpu_master_new.head())
# verify unique hosts
cpu_master_new["hostName"].unique()

,id,ts,host_id,cpu_usage_pct,memory_usage_pct,power_kw,temperature_c,status,ilo_server_id,inlet_temp_c,cpu_temp_c,processor_count,hostName
0,1,2026-04-02 12:22:00+00:00,1,4.0,74.0,0.220,20.0,Normal,1,20.0,38.0,2,10.10.10.65
1,2,2026-04-02 12:22:00+00:00,2,17.0,70.0,0.245,20.0,Normal,2,20.0,43.0,2,10.10.10.150
2,3,2026-04-02 12:22:00+00:00,3,28.0,27.0,0.253,18.0,Normal,3,18.0,28.0,2,10.10.10.2
3,4,2026-04-02 12:24:00+00:00,1,5.0,74.0,0.220,20.0,Normal,1,20.0,38.0,2,10.10.10.65
4,5,2026-04-02 12:24:00+00:00,2,16.0,70.0,0.263,20.0,Normal,2,20.0,44.0,2,10.10.10.150


array(['10.10.10.65', '10.10.10.150', '10.10.10.2'], dtype=object)

In [10]:
# CPU Dataset information
cpu_master_new.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 92792 entries, 0 to 92791
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype              
---  ------            --------------  -----              
 0   id                92792 non-null  int64              
 1   ts                92792 non-null  datetime64[ns, UTC]
 2   host_id           92792 non-null  int64              
 3   cpu_usage_pct     92792 non-null  float64            
 4   memory_usage_pct  92792 non-null  float64            
 5   power_kw          92792 non-null  float64            
 6   temperature_c     92792 non-null  float64            
 7   status            92792 non-null  object             
 8   ilo_server_id     92792 non-null  int64              
 9   inlet_temp_c      92792 non-null  float64            
 10  cpu_temp_c        92792 non-null  float64            
 11  processor_count   92792 non-null  int64              
 12  hostName          92792 non-null  object             
dtypes

In [11]:
# missing values check
cpu_master_new.isnull().sum()

id                  0
ts                  0
host_id             0
cpu_usage_pct       0
memory_usage_pct    0
power_kw            0
temperature_c       0
status              0
ilo_server_id       0
inlet_temp_c        0
cpu_temp_c          0
processor_count     0
hostName            0
dtype: int64

In [12]:
# duplicate check
cpu_master_new.duplicated().sum()

np.int64(0)

In [13]:
# timestamp range (before filtering)
print("Earliest Timestamp:")
print(cpu_master_new["ts"].min())

print("\nLatest Timestamp:")
print(cpu_master_new["ts"].max())

Earliest Timestamp:
2026-04-02 12:22:00+00:00

Latest Timestamp:
2026-07-08 04:18:00+00:00


In [14]:
# keep only data from 12th June 2026 onward
CUTOFF_DATE = "2026-06-12"

cpu_master_new = cpu_master_new[
    cpu_master_new["ts"] >= CUTOFF_DATE
].copy()

print(cpu_master_new.shape)
print("\nNew earliest timestamp:", cpu_master_new["ts"].min())
print("New latest timestamp:", cpu_master_new["ts"].max())

(17805, 13)

New earliest timestamp: 2026-06-12 00:01:00+00:00
New latest timestamp: 2026-07-08 04:18:00+00:00


In [15]:
# Sort dataset
cpu_master = cpu_master_new.sort_values(
    ["hostName", "ts"]
)
# reset index
cpu_master = cpu_master.reset_index(
    drop=True
)

In [16]:
# save processed dataset
Path("../data/Processed").mkdir(
    parents=True,
    exist_ok=True
)
cpu_master.to_csv(
    "../data/inference_data/cpu_master_new.csv",
    index=False
)
print("✓ Processed dataset saved successfully.")

✓ Processed dataset saved successfully.


In [17]:
print(cpu_master.shape)
cpu_master.head()

(17805, 13)


,id,ts,host_id,cpu_usage_pct,memory_usage_pct,power_kw,temperature_c,status,ilo_server_id,inlet_temp_c,cpu_temp_c,processor_count,hostName
0,74990,2026-06-12 00:01:00+00:00,2,1.28,21.65,0.256,25.0,Normal,2,25.0,44.0,2,10.10.10.150
1,74993,2026-06-12 00:06:00+00:00,2,1.92,21.65,0.257,27.0,Normal,2,27.0,45.0,2,10.10.10.150
2,74996,2026-06-12 00:11:00+00:00,2,1.23,21.65,0.210,26.0,Normal,2,26.0,44.0,2,10.10.10.150
3,74999,2026-06-12 00:16:00+00:00,2,2.23,21.65,0.208,21.0,Normal,2,21.0,42.0,2,10.10.10.150
4,75002,2026-06-12 00:21:00+00:00,2,1.33,21.65,0.210,23.0,Normal,2,23.0,42.0,2,10.10.10.150
